# Spatial Diagnostic Tests: Detecting and Modeling Spatial Dependence

**Version**: 1.0  
**Date**: 2026-02-22  
**Estimated Duration**: 120 minutes  
**Difficulty**: Intermediate-Advanced

## Learning Objectives
1. **Understand** spatial dependence and why it matters in panel data
2. **Distinguish** between SAR (spatial lag) and SEM (spatial error) models
3. **Construct** spatial weight matrices from geographic data
4. **Apply** LM tests to detect spatial dependence (lag, error, robust)
5. **Use** the LM test decision tree to choose appropriate model specification
6. **Calculate** and interpret Moran's I for global spatial autocorrelation
7. **Apply** LISA (Local Indicators of Spatial Association) to detect spatial clusters
8. **Visualize** spatial patterns using maps and Moran scatterplots
9. **Make** informed decisions about spatial model specification

## Prerequisites
- Understanding of spatial econometric models (SAR, SEM)
- Familiarity with spatial weight matrices
- Completion of static panel models tutorials

## Datasets
- US County Economic Panel (200 counties, 10 years)
- EU Regional Panel (100 regions, 15 years)

## Estimated Time per Section
| Section | Topic | Time |
|---------|-------|------|
| 1 | What is Spatial Dependence? (Theory) | 20 min |
| 2 | LM Tests for Spatial Dependence | 25 min |
| 3 | Moran's I | 20 min |
| 4 | LISA -- Local Indicators of Spatial Association | 20 min |
| 5 | Practical Applications | 20 min |
| 6 | Exercises | 15 min |

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")


# PanelBox imports
# Local utilities
import sys

import panelbox
from panelbox.datasets import load_dataset
from panelbox.diagnostics.spatial_tests import (
    LocalMoranI,
    MoranIPanelTest,
    lm_error_test,
    lm_lag_test,
    robust_lm_error_test,
    robust_lm_lag_test,
    run_lm_tests,
)
from panelbox.models.static import PooledOLS

BASE_DIR = Path("..")
sys.path.insert(0, str(BASE_DIR / "utils"))
from spatial_helpers import (
    build_weight_matrix,
    lm_decision_tree_summary,
    plot_lisa_map,
    plot_moran_scatterplot,
)

# Configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
np.random.seed(42)

# Paths
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "spatial"
TABLES_DIR = OUTPUT_DIR / "tables" / "spatial"
RESULTS_DIR = OUTPUT_DIR / "results" / "spatial"

for d in [FIGURES_DIR, TABLES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"PanelBox version: {panelbox.__version__}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

---

## Section 1: What is Spatial Dependence?

### 1.1 Tobler's First Law of Geography

> "Everything is related to everything else, but near things are more related than distant things."
> -- Waldo Tobler (1970)

**Formal definition**: Spatial dependence exists when:

$$\text{Cov}(y_i, y_j) \neq 0 \quad \text{for } i \neq j, \text{ where } i \text{ and } j \text{ are spatial neighbors}$$

**Why it matters for econometrics:**
1. **Violates OLS assumptions**: Errors not independent $\Rightarrow$ biased standard errors
2. **Economic interpretation**: Spillovers, peer effects, common shocks
3. **Omitted variable bias**: Spatial effects not modeled explicitly can bias coefficients
4. **Policy implications**: Interventions in one region affect neighboring regions

Let's visualize the difference between spatially random and spatially dependent data.

In [ ]:
# Visual demonstration: random vs spatially dependent data
np.random.seed(42)
N = 100  # 10x10 grid
side = 10

# Create grid coordinates
x_coords = np.array([i % side for i in range(N)])
y_coords = np.array([i // side for i in range(N)])

# Build a queen contiguity W for the grid
W_grid = np.zeros((N, N))
for i in range(N):
    ri, ci = i // side, i % side
    for j in range(N):
        rj, cj = j // side, j % side
        if i != j and abs(ri - rj) <= 1 and abs(ci - cj) <= 1:
            W_grid[i, j] = 1.0
row_sums = W_grid.sum(axis=1)
row_sums[row_sums == 0] = 1
W_grid = W_grid / row_sums[:, np.newaxis]

# Generate spatially dependent data via SAR: y = (I - rho*W)^{-1} * eps
rho_demo = 0.7
eps = np.random.randn(N)
A = np.eye(N) - rho_demo * W_grid
y_spatial = np.linalg.solve(A, eps)

# Independent data
y_random = np.random.randn(N)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sc1 = ax1.scatter(
    x_coords, y_coords, c=y_random, cmap="RdBu_r", s=150, edgecolors="white", linewidths=0.5
)
ax1.set_title("No Spatial Dependence (random)", fontsize=14, fontweight="bold")
ax1.set_xlabel("X coordinate")
ax1.set_ylabel("Y coordinate")
plt.colorbar(sc1, ax=ax1, label="Value")

sc2 = ax2.scatter(
    x_coords, y_coords, c=y_spatial, cmap="RdBu_r", s=150, edgecolors="white", linewidths=0.5
)
ax2.set_title(f"With Spatial Dependence (rho={rho_demo})", fontsize=14, fontweight="bold")
ax2.set_xlabel("X coordinate")
ax2.set_ylabel("Y coordinate")
plt.colorbar(sc2, ax=ax2, label="Value")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_spatial_dependence_demo.png", dpi=300, bbox_inches="tight")
plt.show()

print("Notice how the right panel shows spatial clustering:")
print("  - Similar values (colors) tend to be near each other")
print("  - This is characteristic of positive spatial autocorrelation")

### 1.2 SAR vs SEM: Two Types of Spatial Dependence

#### Spatial AutoRegressive (SAR) Model

$$y = \rho \cdot W \cdot y + X \cdot \beta + \varepsilon$$

- $\rho$: spatial autoregressive parameter
- $W \cdot y$: weighted average of neighbors' outcomes
- **Substantive dependence**: $y_i$ depends directly on neighbors' $y_j$
- **Examples**: Technology spillovers, tax competition, housing price contagion
- **Key feature**: Feedback loops ($y_i \rightarrow y_j \rightarrow y_i$)

#### Spatial Error Model (SEM)

$$y = X \cdot \beta + u, \quad u = \lambda \cdot W \cdot u + \varepsilon$$

- $\lambda$: spatial error parameter
- $W \cdot u$: spatial correlation in errors (unobserved shocks)
- **Nuisance dependence**: Unobserved common factors are spatially correlated
- **Examples**: Weather shocks, shared amenities, measurement error with spatial pattern
- **Key feature**: No direct feedback (just correlated disturbances)

#### How to Choose?

| Aspect | SAR | SEM |
|--------|-----|-----|
| **Economic story** | Direct spillovers | Common unobserved factors |
| **Coefficients** | Biased if omitted | Unbiased but inefficient |
| **Standard errors** | Biased | Biased |
| **Detection** | LM-Lag test | LM-Error test |

The key takeaway: **both forms of spatial dependence invalidate standard inference**, but for different reasons.

### 1.3 Spatial Weight Matrix (W)

The **spatial weight matrix** $W$ is an $N \times N$ matrix that defines the spatial relationships between units. Its construction is a critical modeling choice.

**Common construction methods:**

| Method | Definition | Use case |
|--------|-----------|----------|
| **Contiguity** | $w_{ij} = 1$ if $i,j$ share a border | Administrative regions |
| **Distance** | $w_{ij} = 1/d_{ij}$ if $d_{ij} < \text{threshold}$ | Points in space |
| **K-nearest neighbors** | $w_{ij} = 1$ if $j$ is among $k$ nearest to $i$ | Irregularly spaced units |
| **Economic distance** | $w_{ij} = f(|\text{GDP}_i - \text{GDP}_j|)$ | Economic similarity |

**Row normalization** (standard practice):
$$w^*_{ij} = \frac{w_{ij}}{\sum_k w_{ik}}$$

This ensures each row sums to 1, so $Wy$ is a weighted **average** of neighbors' values.

Let's construct weight matrices from our US county data.

In [ ]:
# Load US county data
counties = load_dataset("us_counties", category="diagnostics")
coords_us = load_dataset("coordinates_us", category="diagnostics")
W_counties = np.load(DATA_DIR / "spatial" / "W_counties.npy")

print("US County Economic Panel")
print("=" * 60)
print(f"Shape: {counties.shape}")
print(f"Counties: {counties.county_id.nunique()}")
print(f"Years: {counties.year.min()} - {counties.year.max()}")
print(f"\nVariables: {list(counties.columns)}")
print(f"\nW matrix shape: {W_counties.shape}")
print(f"W row sums (first 5): {W_counties.sum(axis=1)[:5]}")
print(f"W diagonal (should be 0): {W_counties.diagonal().sum()}")
print(f"W non-zero entries: {(W_counties > 0).sum()} / {W_counties.size}")
print(f"W sparsity: {1 - (W_counties > 0).sum() / W_counties.size:.2%}")
print("\nFirst few rows:")
display(counties.head())
print("\nSummary statistics:")
display(counties.describe())

In [ ]:
# Build K-nearest neighbor weight matrix using spatial helpers
coords_array = coords_us[["latitude", "longitude"]].values

W_knn5 = build_weight_matrix(coords_array, method="knn", k=5)
W_knn8 = build_weight_matrix(coords_array, method="knn", k=8)
W_dist = build_weight_matrix(coords_array, method="distance")

print("Weight Matrix Comparison")
print("=" * 60)
for name, W in [
    ("Queen contiguity (pre-built)", W_counties),
    ("KNN (k=5)", W_knn5),
    ("KNN (k=8)", W_knn8),
    ("Distance band", W_dist),
]:
    n_neighbors = (W > 0).sum(axis=1)
    print(f"\n{name}:")
    print(f"  Avg neighbors: {n_neighbors.mean():.1f}")
    print(f"  Min neighbors: {n_neighbors.min()}")
    print(f"  Max neighbors: {n_neighbors.max()}")
    print(f"  Row sums = 1? {np.allclose(W.sum(axis=1), 1.0)}")

In [ ]:
# Visualize the spatial structure: counties on a map
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Map of counties colored by unemployment (2019)
df_2019 = counties[counties["year"] == 2019].merge(coords_us, on="county_id")

sc1 = ax1.scatter(
    df_2019["longitude"],
    df_2019["latitude"],
    c=df_2019["unemployment"],
    cmap="YlOrRd",
    s=30,
    edgecolors="grey",
    linewidths=0.3,
    alpha=0.8,
)
ax1.set_title("Unemployment Rate by County (2019)", fontsize=13, fontweight="bold")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1, label="Unemployment Rate")

# Map colored by income
sc2 = ax2.scatter(
    df_2019["longitude"],
    df_2019["latitude"],
    c=df_2019["log_income"],
    cmap="viridis",
    s=30,
    edgecolors="grey",
    linewidths=0.3,
    alpha=0.8,
)
ax2.set_title("Log Income by County (2019)", fontsize=13, fontweight="bold")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2, label="Log Income")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_county_maps.png", dpi=300, bbox_inches="tight")
plt.show()

print("Visual inspection suggests spatial clustering in both variables.")
print("We will formally test this using Moran's I and LM tests below.")

---

## Section 2: LM Tests for Spatial Dependence

### 2.1 Overview

**Lagrange Multiplier (LM) tests** detect spatial dependence **without estimating a spatial model**. The procedure is:
1. Estimate a standard OLS/panel model (ignoring spatial effects)
2. Extract residuals
3. Compute LM statistics using residuals and $W$
4. Compare to $\chi^2$ critical values

**Four main tests** (Anselin 1988; Anselin et al. 1996):

| Test | H0 | Detects |
|------|----|---------|
| LM-Lag | $\rho = 0$ | SAR (spatial lag) |
| LM-Error | $\lambda = 0$ | SEM (spatial error) |
| Robust LM-Lag | $\rho = 0$ (conditional on error) | SAR, robust to SEM |
| Robust LM-Error | $\lambda = 0$ (conditional on lag) | SEM, robust to SAR |

All test statistics follow $\chi^2(1)$ under the null.

In [ ]:
# Step 1: Estimate OLS model (ignoring spatial effects)
df_panel = counties.sort_values(["county_id", "year"]).reset_index(drop=True)

model_ols = PooledOLS(
    "unemployment ~ log_income + log_population + manufacturing_share",
    df_panel,
    entity_col="county_id",
    time_col="year",
)
ols_result = model_ols.fit()

print("Pooled OLS Baseline (ignoring spatial dependence)")
print("=" * 60)
print(ols_result.summary())

### 2.2 Running Individual LM Tests

The LM test functions take **residuals**, **design matrix X**, and **weight matrix W** as arguments. Note that $W$ is $N \times N$ (cross-sectional dimension only); the functions internally expand it to $NT \times NT$ for panel data using the Kronecker product $I_T \otimes W$.

In [ ]:
# Extract residuals and design matrix from OLS
residuals = ols_result.resid
X = ols_result.model.exog

print(
    f"Residuals: {residuals.shape} (N*T = {counties.county_id.nunique()} * {counties.year.nunique()})"
)
print(f"X matrix: {X.shape}")
print(f"W matrix: {W_counties.shape} (N x N)")

# LM-Lag test: H0: rho = 0
print("\n" + "=" * 60)
result_lag = lm_lag_test(residuals, X, W_counties)
print(result_lag.summary())

# LM-Error test: H0: lambda = 0
print("\n" + "=" * 60)
result_error = lm_error_test(residuals, X, W_counties)
print(result_error.summary())

# Robust LM-Lag: H0: rho = 0 (controlling for lambda)
print("\n" + "=" * 60)
result_robust_lag = robust_lm_lag_test(residuals, X, W_counties)
print(result_robust_lag.summary())

# Robust LM-Error: H0: lambda = 0 (controlling for rho)
print("\n" + "=" * 60)
result_robust_error = robust_lm_error_test(residuals, X, W_counties)
print(result_robust_error.summary())

### 2.3 The Decision Tree (Anselin & Rey 2014)

The `run_lm_tests()` function runs all four tests and provides a model recommendation using the classical decision tree:

```
Step 1: Run LM-Lag and LM-Error

Case 1: Neither significant  -->  No spatial dependence  -->  OLS is fine
Case 2: Only LM-Lag significant  -->  Use SAR model
Case 3: Only LM-Error significant  -->  Use SEM model
Case 4: BOTH significant  -->  Run Robust LM tests
  4a: Only Robust LM-Lag significant  -->  SAR
  4b: Only Robust LM-Error significant  -->  SEM
  4c: Both robust significant  -->  SDM (Spatial Durbin Model)
  4d: Neither robust significant  -->  Use larger standard LM as tiebreaker
```

In [ ]:
# Automated decision tree using run_lm_tests()
lm_results = run_lm_tests(ols_result, W_counties, alpha=0.05)

print("LM Test Summary Table")
print("=" * 60)
display(lm_results["summary"])

print(f"\nRecommended model: {lm_results['recommendation']}")
print(f"Reason: {lm_results['reason']}")

# Save results
lm_results["summary"].to_csv(TABLES_DIR / "04_lm_test_results_us.csv", index=False)

In [ ]:
# Detailed decision tree walkthrough
tree_summary = lm_decision_tree_summary(lm_results, alpha=0.05)
print(tree_summary)

In [ ]:
# Compare results across different weight matrices
print("Sensitivity of LM Tests to Weight Matrix Choice")
print("=" * 60)

w_matrices = {
    "Queen contiguity": W_counties,
    "KNN (k=5)": W_knn5,
    "KNN (k=8)": W_knn8,
}

rows = []
for w_name, W in w_matrices.items():
    try:
        res = run_lm_tests(ols_result, W, alpha=0.05)
        rows.append(
            {
                "W matrix": w_name,
                "LM-Lag stat": res["lm_lag"].statistic,
                "LM-Lag p": res["lm_lag"].pvalue,
                "LM-Error stat": res["lm_error"].statistic,
                "LM-Error p": res["lm_error"].pvalue,
                "Recommendation": res["recommendation"],
            }
        )
    except Exception as e:
        print(f"  {w_name}: Error -- {e}")

sensitivity_df = pd.DataFrame(rows)
display(sensitivity_df)

sensitivity_df.to_csv(TABLES_DIR / "04_lm_sensitivity_W.csv", index=False)
print("\nNote: Results may vary with W choice -- this is expected.")
print("Always report results for multiple W specifications as a robustness check.")

---

## Section 3: Moran's I

### 3.1 Global Moran's I

**Purpose**: Measure overall spatial autocorrelation -- are similar values clustered?

**Formula**:
$$I = \frac{N}{S_0} \cdot \frac{\sum_i \sum_j w_{ij} (y_i - \bar{y})(y_j - \bar{y})}{\sum_i (y_i - \bar{y})^2}$$

where $S_0 = \sum_i \sum_j w_{ij}$.

**Interpretation**:
- $I > 0$: Positive spatial autocorrelation (similar values cluster)
- $I \approx 0$: No spatial pattern (random)
- $I < 0$: Negative spatial autocorrelation (dissimilar values cluster, checkerboard)

**Expected value under randomization**: $E[I] = -1/(N-1) \approx 0$ for large $N$

In [ ]:
# Moran's I on OLS residuals -- pooled across time
moran_test = MoranIPanelTest(
    residuals=ols_result.resid,
    W=W_counties,
    entity_ids=df_panel["county_id"].values,
    time_ids=df_panel["year"].values,
)

# Pooled method: average residuals by entity, then compute I
pooled_result = moran_test.run(method="pooled")
print("Moran's I (Pooled Residuals)")
print("=" * 60)
print(pooled_result.summary())
print()

# Average across periods
avg_result = moran_test.run(method="average")
print("Moran's I (Average Across Periods)")
print("=" * 60)
print(avg_result.summary())

In [ ]:
# Moran's I directly on unemployment values (not residuals) for 2019
df_2019 = df_panel[df_panel["year"] == 2019].copy()
unemp_values = df_2019["unemployment"].values

# Use MoranIPanelTest on a single cross-section
moran_cs = MoranIPanelTest(
    residuals=unemp_values,
    W=W_counties,
    entity_ids=df_2019["county_id"].values,
    time_ids=df_2019["year"].values,
)
cs_result = moran_cs.run(method="pooled")

print("Moran's I on Raw Unemployment (2019)")
print("=" * 60)
print(f"I = {cs_result.statistic:.4f}")
print(f"E[I] = {cs_result.expected_value:.4f}")
print(f"Z-score = {cs_result.z_score:.4f}")
print(f"P-value = {cs_result.pvalue:.4f}")
print(f"Conclusion: {cs_result.conclusion}")
print()
if cs_result.pvalue < 0.05:
    print("Interpretation: Significant positive spatial autocorrelation.")
    print("Unemployment rates are spatially clustered -- nearby counties have similar rates.")
else:
    print("Interpretation: No significant spatial autocorrelation at 5% level.")
    print("Note: This does not mean no spatial effects -- the signal may be subtle.")

In [ ]:
# Moran's I over time: evolution of spatial autocorrelation
by_period = moran_test.run(method="by_period")

years = sorted(by_period.keys())
I_values = [by_period[y].statistic for y in years]
p_values = [by_period[y].pvalue for y in years]
years_int = [int(y) for y in years]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax1.plot(years_int, I_values, marker="o", linewidth=2, color="steelblue")
ax1.axhline(
    pooled_result.expected_value,
    color="red",
    linestyle="--",
    alpha=0.5,
    label=f"E[I] = {pooled_result.expected_value:.4f}",
)
ax1.fill_between(years_int, pooled_result.expected_value, I_values, alpha=0.2)
ax1.set_ylabel("Moran's I", fontsize=12)
ax1.set_title(
    "Evolution of Spatial Autocorrelation in Residuals Over Time", fontsize=14, fontweight="bold"
)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

ax2.bar(years_int, p_values, color=["forestgreen" if p < 0.05 else "lightgrey" for p in p_values])
ax2.axhline(0.05, color="red", linestyle="--", alpha=0.7, label="alpha = 0.05")
ax2.set_xlabel("Year", fontsize=12)
ax2.set_ylabel("P-value", fontsize=12)
ax2.set_title("Significance of Spatial Autocorrelation by Year", fontsize=14, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_morans_i_over_time.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary
moran_time_df = pd.DataFrame({"year": years_int, "morans_i": I_values, "pvalue": p_values})
print("Moran's I by Year:")
display(moran_time_df)
moran_time_df.to_csv(TABLES_DIR / "04_morans_i_by_year.csv", index=False)

### 3.2 Moran Scatterplot

The **Moran scatterplot** is a powerful visual tool that plots the standardized variable $z_i$ against its spatial lag $Wz_i$.

**Quadrants**:
- **HH** (upper-right): High values surrounded by high values -- spatial clusters
- **LL** (lower-left): Low values surrounded by low values -- spatial clusters
- **HL** (lower-right): High values surrounded by low values -- spatial outliers
- **LH** (upper-left): Low values surrounded by high values -- spatial outliers

The **slope of the regression line** approximates Moran's I.

In [ ]:
# Moran scatterplot for unemployment (2019)
df_2019 = df_panel[df_panel["year"] == 2019].copy()

fig = plot_moran_scatterplot(
    values=df_2019["unemployment"].values,
    W=W_counties,
    entity_ids=df_2019["county_id"].values,
    variable_name="Unemployment",
    annotate_outliers=True,
    save_path=str(FIGURES_DIR / "04_moran_scatter_unemployment.png"),
)
plt.show()

print("Interpretation:")
print("  - Points in HH/LL quadrants indicate spatial clustering")
print("  - Points in HL/LH quadrants are spatial outliers")
print("  - The slope of the line approximates Moran's I")

In [ ]:
# Moran scatterplot for log income
fig = plot_moran_scatterplot(
    values=df_2019["log_income"].values,
    W=W_counties,
    entity_ids=df_2019["county_id"].values,
    variable_name="Log Income",
    annotate_outliers=True,
    save_path=str(FIGURES_DIR / "04_moran_scatter_income.png"),
)
plt.show()

---

## Section 4: LISA -- Local Indicators of Spatial Association

### 4.1 Concept

**Global Moran's I** gives a single summary for the entire study area. But we often want to know **where** clusters and outliers are located.

**LISA** (Anselin 1995) decomposes the global Moran's I into location-specific measures:

$$I_i = \frac{(y_i - \bar{y})}{\sigma^2} \sum_j w_{ij} (y_j - \bar{y})$$

**Property**: $\sum_i I_i \propto \text{Global Moran's I}$

### 4.2 Cluster Types

| Type | Description | Color |
|------|------------|-------|
| **HH** | Hot spot: high value, high neighbors | Red |
| **LL** | Cold spot: low value, low neighbors | Blue |
| **HL** | High outlier: high value, low neighbors | Orange |
| **LH** | Low outlier: low value, high neighbors | Light blue |
| **NS** | Not significant | Grey |

### 4.3 Significance Testing

LISA uses **permutation inference**:
1. Compute observed $I_i$ for each location
2. Randomly permute values 999 times
3. Compute reference distribution of $I_i$
4. P-value = fraction of permutations more extreme than observed

**Warning**: LISA with many permutations can be slow. Start with 99 for exploration, use 999 for final results.

In [ ]:
# LISA analysis for unemployment (2019)
print("Computing LISA (Local Moran's I) -- this may take a moment...")

df_2019 = df_panel[df_panel["year"] == 2019].copy()
values_2019 = df_2019["unemployment"].values

lisa = LocalMoranI(
    values=values_2019,
    W=W_counties,
    entity_ids=df_2019["county_id"].values,
)
lisa_result = lisa.run(permutations=499)  # Use 499 for reasonable speed

print("\nLISA Results")
print("=" * 60)
print(lisa_result.summary())

# Get cluster classification
clusters = lisa_result.get_clusters(alpha=0.05)
print("\nCluster Distribution:")
cluster_counts = clusters["cluster_type"].value_counts()
print(cluster_counts)

# Save
clusters.to_csv(TABLES_DIR / "04_lisa_clusters_unemployment.csv", index=False)

In [ ]:
# LISA cluster map
coords_array = coords_us[["latitude", "longitude"]].values

fig = plot_lisa_map(
    local_i=lisa_result.local_i,
    pvalues=lisa_result.pvalues,
    z_values=lisa_result.z_values,
    Wz_values=lisa_result.Wz_values,
    coordinates=coords_array,
    alpha=0.05,
    save_path=str(FIGURES_DIR / "04_lisa_cluster_map_unemployment.png"),
)
plt.show()

print("LISA Cluster Map Interpretation:")
print("  - Red (HH): High unemployment clusters (hot spots)")
print("  - Blue (LL): Low unemployment clusters (cold spots)")
print("  - Orange (HL): High unemployment surrounded by low (outliers)")
print("  - Light blue (LH): Low unemployment surrounded by high (outliers)")
print("  - Grey (NS): Not significant at alpha = 0.05")

In [ ]:
# Detailed look at significant clusters
sig_clusters = clusters[clusters["cluster_type"] != "Not significant"].copy()
sig_clusters = sig_clusters.merge(
    df_2019[["county_id", "unemployment", "log_income", "manufacturing_share"]],
    left_on="entity_id",
    right_on="county_id",
    how="left",
)

print("Significant LISA Clusters")
print("=" * 60)

for ctype in ["HH", "LL", "HL", "LH"]:
    subset = sig_clusters[sig_clusters["cluster_type"] == ctype]
    if len(subset) > 0:
        print(f"\n{ctype} Clusters ({len(subset)} counties):")
        print(f"  Mean unemployment: {subset['unemployment'].mean():.4f}")
        print(f"  Mean log income: {subset['log_income'].mean():.4f}")
        print(f"  Mean manufacturing share: {subset['manufacturing_share'].mean():.4f}")

# Summary statistics by cluster type
if len(sig_clusters) > 0:
    print("\nCluster Characteristics Summary:")
    summary_by_cluster = (
        sig_clusters.groupby("cluster_type")
        .agg(
            {
                "unemployment": ["mean", "std", "count"],
                "log_income": "mean",
                "manufacturing_share": "mean",
            }
        )
        .round(4)
    )
    display(summary_by_cluster)
    summary_by_cluster.to_csv(TABLES_DIR / "04_cluster_characteristics.csv")

---

## Section 5: Practical Applications

### 5.1 Application 1: EU Regional GDP -- Spatial Error Analysis

The EU regions dataset has a spatial error structure (SEM) built into the DGP. Let's see if our tests detect it.

In [ ]:
# Load EU regions data
eu_regions = load_dataset("eu_regions", category="diagnostics")
coords_eu = load_dataset("coordinates_eu", category="diagnostics")
W_eu = np.load(DATA_DIR / "spatial" / "W_eu_contiguity.npy")

print("EU Regional Panel")
print("=" * 60)
print(f"Shape: {eu_regions.shape}")
print(f"Regions: {eu_regions.region_id.nunique()}")
print(f"Countries: {eu_regions.country.nunique()}")
print(f"Years: {eu_regions.year.min()} - {eu_regions.year.max()}")
print(f"\nVariables: {list(eu_regions.columns)}")
print(f"W shape: {W_eu.shape}")
display(eu_regions.head())

In [ ]:
# Moran's I over time for EU GDP
eu_sorted = eu_regions.sort_values(["region_id", "year"]).reset_index(drop=True)

# OLS baseline
eu_model = PooledOLS(
    "log_gdp_pc ~ fdi + rd_expenditure + infrastructure",
    eu_sorted,
    entity_col="region_id",
    time_col="year",
)
eu_result = eu_model.fit()

# Moran's I by period
eu_moran = MoranIPanelTest(
    residuals=eu_result.resid,
    W=W_eu,
    entity_ids=eu_sorted["region_id"].values,
    time_ids=eu_sorted["year"].values,
)

eu_by_period = eu_moran.run(method="by_period")

eu_years = sorted(eu_by_period.keys())
eu_I_vals = [eu_by_period[y].statistic for y in eu_years]
eu_p_vals = [eu_by_period[y].pvalue for y in eu_years]
eu_years_int = [int(y) for y in eu_years]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eu_years_int, eu_I_vals, marker="o", linewidth=2, color="steelblue")
ax.axhline(0, color="red", linestyle="--", alpha=0.5)
ax.fill_between(eu_years_int, 0, eu_I_vals, alpha=0.2)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Moran's I", fontsize=12)
ax.set_title(
    "Spatial Autocorrelation of GDP Residuals in EU Regions", fontsize=14, fontweight="bold"
)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_eu_morans_i_evolution.png", dpi=300, bbox_inches="tight")
plt.show()

eu_moran_df = pd.DataFrame({"year": eu_years_int, "morans_i": eu_I_vals, "pvalue": eu_p_vals})
display(eu_moran_df)
eu_moran_df.to_csv(TABLES_DIR / "04_eu_morans_i_by_year.csv", index=False)

In [ ]:
# LM tests for EU data
eu_lm = run_lm_tests(eu_result, W_eu, alpha=0.05)

print("LM Tests for EU Regional GDP")
print("=" * 60)
display(eu_lm["summary"])
print(f"\nRecommendation: {eu_lm['recommendation']}")
print(f"Reason: {eu_lm['reason']}")

# Decision tree
print()
print(lm_decision_tree_summary(eu_lm, alpha=0.05))

eu_lm["summary"].to_csv(TABLES_DIR / "04_lm_test_results_eu.csv", index=False)

### 5.2 Application 2: Spatial Model Estimation

Based on the LM test results, let's estimate the recommended spatial model and compare with OLS.

In [ ]:
# Estimate spatial models based on LM test recommendation
recommendation = lm_results["recommendation"]
print(f"US Counties -- Recommended model: {recommendation}")
print("=" * 60)

try:
    if "SAR" in recommendation or "SDM" in recommendation or "GNS" in recommendation:
        from panelbox.models.spatial import SpatialLag

        sar = SpatialLag(
            "unemployment ~ log_income + log_population + manufacturing_share",
            df_panel,
            entity_col="county_id",
            time_col="year",
            W=W_counties,
        )
        sar_result = sar.fit(effects="pooled", method="qml")
        print("SAR Model Results:")
        print(sar_result.summary())

    if "SEM" in recommendation or "SDM" in recommendation or "GNS" in recommendation:
        from panelbox.models.spatial import SpatialError

        sem = SpatialError(
            "unemployment ~ log_income + log_population + manufacturing_share",
            df_panel,
            entity_col="county_id",
            time_col="year",
            W=W_counties,
        )
        sem_result = sem.fit(effects="pooled", method="gmm")
        print("\nSEM Model Results:")
        print(sem_result.summary())

    if "No spatial" in recommendation:
        print("No spatial dependence detected -- OLS is appropriate.")
        print(ols_result.summary())

except Exception as e:
    print(f"Spatial model estimation error: {e}")
    print("Falling back to OLS results for comparison.")

# Always show OLS for reference
print("\nOLS Baseline (for comparison):")
print(f"  Coefficients: {dict(ols_result.params)}")
print(f"  R-squared: {ols_result.rsquared:.4f}")

### 5.3 Application 3: LISA for Policy Targeting

LISA analysis can identify **priority regions** for policy intervention by detecting clusters of disadvantage.

In [ ]:
# LISA on EU GDP to identify priority regions
latest_year = eu_regions["year"].max()
eu_latest = eu_sorted[eu_sorted["year"] == latest_year].copy()

print(f"LISA Analysis: EU GDP per Capita ({latest_year})")
print("Computing permutation test (this may take a moment)...")

lisa_eu = LocalMoranI(
    values=eu_latest["gdp_per_capita"].values,
    W=W_eu,
    entity_ids=eu_latest["region_id"].values,
)
lisa_eu_result = lisa_eu.run(permutations=499)

print(lisa_eu_result.summary())

# Get clusters
eu_clusters = lisa_eu_result.get_clusters(alpha=0.05)

# Merge with region data
eu_clusters = eu_clusters.merge(
    eu_latest[["region_id", "country", "gdp_per_capita"]],
    left_on="entity_id",
    right_on="region_id",
    how="left",
)

print("\nCluster Distribution:")
print(eu_clusters["cluster_type"].value_counts())

In [ ]:
# LISA cluster map for EU
eu_coords_array = coords_eu[["latitude", "longitude"]].values

fig = plot_lisa_map(
    local_i=lisa_eu_result.local_i,
    pvalues=lisa_eu_result.pvalues,
    z_values=lisa_eu_result.z_values,
    Wz_values=lisa_eu_result.Wz_values,
    coordinates=eu_coords_array,
    alpha=0.05,
    save_path=str(FIGURES_DIR / "04_lisa_eu_gdp.png"),
)
plt.show()

In [ ]:
# Policy targeting: identify LL clusters (low GDP surrounded by low GDP)
ll_clusters = eu_clusters[eu_clusters["cluster_type"] == "LL"].copy()
hh_clusters = eu_clusters[eu_clusters["cluster_type"] == "HH"].copy()

print("Policy Analysis: Priority Regions for Economic Development")
print("=" * 60)

if len(ll_clusters) > 0:
    print(f"\nLL Clusters (Low-GDP regions, priority for intervention): {len(ll_clusters)} regions")
    print(f"  Mean GDP per capita: {ll_clusters['gdp_per_capita'].mean():.0f}")
    print(f"  Countries represented: {ll_clusters['country'].unique().tolist()}")
    print("\n  Top 10 priority regions:")
    display(
        ll_clusters.sort_values("gdp_per_capita")[["entity_id", "country", "gdp_per_capita"]].head(
            10
        )
    )
else:
    print("No significant LL clusters detected at alpha = 0.05.")

if len(hh_clusters) > 0:
    print(f"\nHH Clusters (Prosperous regions): {len(hh_clusters)} regions")
    print(f"  Mean GDP per capita: {hh_clusters['gdp_per_capita'].mean():.0f}")
    print(f"  Countries represented: {hh_clusters['country'].unique().tolist()}")

print("\nPolicy Recommendations:")
print("  - Target LL cluster regions with direct investment")
print("  - Leverage spatial spillovers from neighboring HH regions")
print("  - Cross-border cooperation programs for regions near HH/LL boundaries")

---

## Section 6: Exercises

Solutions for Exercises 1-3 are in `../solutions/04_spatial_solutions.ipynb`. Exercise 4 is independent.

### Exercise 1: Build Weight Matrix and Compute Moran's I (Easy)

**Task**: Construct a spatial weight matrix and test for spatial autocorrelation in income.

**Steps**:
1. Load `coordinates_us.csv` and build a KNN weight matrix with $k=5$
2. Compute Moran's I for `log_income` using the 2019 cross-section
3. Create a Moran scatterplot
4. Interpret results: is income spatially clustered?

**Expected output**: Weight matrix summary (dimensions, sparsity), Moran's I with z-score and p-value, Moran scatterplot with quadrant labels.

In [ ]:
# Exercise 1: Build Weight Matrix and Compute Moran's I
# ========================================================

# Step 1: Load coordinates and build KNN W
# coords = load_dataset("coordinates_us", category="diagnostics")
# W_ex1 = build_weight_matrix(coords[['latitude', 'longitude']].values, method='knn', k=5)
# print(f'W shape: {W_ex1.shape}')

# Step 2: Compute Moran's I for log_income in 2019
# YOUR CODE HERE

# Step 3: Moran scatterplot
# YOUR CODE HERE

# Step 4: Interpretation
# YOUR ANSWER HERE

### Exercise 2: LM Test Decision Tree (Medium)

**Task**: Apply the full LM test battery to determine the appropriate spatial model for housing/income data.

**Steps**:
1. Using the US counties data, estimate OLS with `log_income` as dependent variable and `education_pct`, `manufacturing_share` as regressors
2. Run all four LM tests using both `W_counties` and a KNN weight matrix
3. Walk through the decision tree for each W specification
4. If SAR is recommended, estimate the spatial lag model and compare coefficients with OLS

**Questions**: Which model is recommended? Does the recommendation change with W?

In [ ]:
# Exercise 2: LM Test Decision Tree
# ===================================

# Step 1: OLS with log_income as dependent variable
# model_ex2 = PooledOLS('log_income ~ education_pct + manufacturing_share',
#                        df_panel, 'county_id', 'year')
# result_ex2 = model_ex2.fit()

# Step 2: Run all four LM tests
# YOUR CODE HERE

# Step 3: Decision tree walkthrough
# YOUR CODE HERE

# Step 4: Estimate spatial model if recommended
# YOUR CODE HERE

### Exercise 3: LISA Cluster Analysis (Hard)

**Task**: Perform a complete LISA cluster analysis on EU regional data.

**Steps**:
1. Compute LISA for `rd_expenditure` (R&D expenditure) using the latest year
2. Identify all four cluster types (HH, LL, HL, LH)
3. Create a LISA cluster map
4. Compare the characteristics of HH vs LL clusters (GDP, infrastructure)
5. Provide an economic interpretation: where are R&D clusters?

**Deliverables**: LISA cluster map, table comparing cluster types, economic interpretation.

In [ ]:
# Exercise 3: LISA Cluster Analysis
# ===================================

# Step 1: LISA for rd_expenditure
# YOUR CODE HERE

# Step 2: Identify cluster types
# YOUR CODE HERE

# Step 3: Cluster map
# YOUR CODE HERE

# Step 4: Compare HH vs LL characteristics
# YOUR CODE HERE

# Step 5: Economic interpretation
# YOUR ANSWER HERE

### Exercise 4: Panel Spatial Analysis (Independent -- No Solution)

**Task**: Perform a complete spatial analysis using the EU panel data across multiple years.

**Requirements**:
1. Test for spatial dependence in GDP per capita over time (Moran's I evolution)
2. Apply LM tests for the latest year and for the full panel
3. Perform LISA analysis and identify how clusters evolve
4. Estimate the appropriate spatial model (SAR, SEM, or SDM)
5. Interpret results and discuss policy implications

**Hint**: Compare early-period vs late-period clusters to see if regional convergence or divergence is occurring.

In [ ]:
# Exercise 4: Panel Spatial Analysis (Independent -- No Solution)
# ================================================================

# Step 1: Spatial dependence over time
# YOUR CODE HERE

# Step 2: LM tests
# YOUR CODE HERE

# Step 3: LISA analysis
# YOUR CODE HERE

# Step 4: Spatial model estimation
# YOUR CODE HERE

# Step 5: Policy implications
# YOUR ANSWER HERE

---

## Section 7: Key Takeaways

### Conceptual
1. **Spatial dependence** violates the OLS independence assumption, leading to biased standard errors
2. **SAR** (spatial lag) captures direct spillovers; **SEM** (spatial error) captures common unobserved shocks
3. The **weight matrix W** defines spatial structure -- its construction is a critical modeling choice
4. **Moran's I** measures global spatial autocorrelation (positive = clustering, negative = dispersion)
5. **LISA** decomposes the global pattern into local clusters and outliers (HH, LL, HL, LH)

### Methodological
6. **LM tests** detect spatial dependence without estimating a spatial model
7. The **Anselin & Rey decision tree** provides systematic model selection (SAR vs SEM vs SDM)
8. **Robust LM tests** are needed when both lag and error tests are significant
9. **Permutation tests** provide exact inference for LISA significance
10. **Multiple testing correction** (Bonferroni, FDR) is important for LISA with many locations

### Practical
11. Always **visualize** spatial patterns first (maps, Moran scatterplot) before formal testing
12. Use the **LM decision tree**, not theory alone, to choose the spatial model
13. LISA identifies **where** to target policy interventions (LL clusters = priority regions)
14. **Weight matrix choice affects results** -- always report robustness across multiple W specifications
15. Spatial models are needed even if you're only interested in $X \rightarrow y$ (SEs are biased otherwise)

---

## Troubleshooting

### Common Errors and Solutions

**1. "Residuals length must be divisible by W dimension"**
- The LM tests automatically expand $W$ from $N \times N$ to $NT \times NT$
- Ensure your data is sorted by `(entity, time)` and $N$ matches W dimension
- Solution: `df = df.sort_values(['entity_col', 'time_col'])`

**2. Weight matrix not row-normalized**
- LM tests assume row-normalized $W$ (each row sums to 1)
- Check: `np.allclose(W.sum(axis=1), 1.0)`
- Fix: `W = W / W.sum(axis=1, keepdims=True)`

**3. LISA is very slow**
- Start with `permutations=99` for exploration
- Use `permutations=499` or `999` only for final results
- For $N > 500$, consider sparse matrix implementations

**4. All LISA clusters are "Not significant"**
- Moran's I may be low (weak spatial pattern)
- Try fewer permutations first to debug
- Check if the variable has sufficient spatial variation

**5. LM test statistics are very large or very small**
- Check that $W$ diagonal is zero: `assert W.diagonal().sum() == 0`
- Verify data is not duplicated across time periods
- Try a different weight matrix specification

---

## Next Steps

- **Spatial Models Tutorials**: Estimation of SAR, SEM, SDM models with PanelBox
- **Notebook 01: Unit Root Tests**: Testing for non-stationarity in panel data
- **Notebook 02: Cointegration Tests**: Long-run equilibrium relationships
- **Notebook 03: Specification Tests**: Hausman test, J-test for model selection

---

## References

### Seminal Papers
1. **Anselin, L. (1988)**. *Spatial Econometrics: Methods and Models*. Kluwer Academic Publishers.
2. **Anselin, L., Bera, A. K., Florax, R., & Yoon, M. J. (1996)**. "Simple diagnostic tests for spatial dependence". *Regional Science and Urban Economics*, 26(1), 77-104.
3. **Anselin, L. (1995)**. "Local indicators of spatial association -- LISA". *Geographical Analysis*, 27(2), 93-115.
4. **Anselin, L., & Rey, S. J. (2014)**. *Modern Spatial Econometrics in Practice*. GeoDa Press.
5. **LeSage, J., & Pace, R. K. (2009)**. *Introduction to Spatial Econometrics*. CRC Press.

### Textbooks
6. **Elhorst, J. P. (2014)**. *Spatial Econometrics: From Cross-Sectional Data to Spatial Panels*. Springer.
7. **Tobler, W. R. (1970)**. "A computer movie simulating urban growth in the Detroit region". *Economic Geography*, 46, 234-240.